

```
🟦 DAY 6 — 2 HRS

1. GENAI — 60 MIN

□ Hugging Face Models
□ AutoModel
□ Model vs Tokenizer
□ Inference Pipeline

2. DSA — 45 MIN

□ Hashing
□ HashMap / Dictionary
□ 2 Easy problems

3. PRACTICE — 15 MIN

□ Load a pretrained model
□ Run one inference task
```



In [1]:
!pip install --upgrade transformers

#### **1. TEXT-CLASSIFIER**

In [16]:
import transformers
from transformers import pipeline

# Hide download progress bars
logging.set_verbosity_error()

text_classifier = pipeline('text-classification')

text = text_classifier("This Hugging Face course is amazing!")
print(text)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998809099197388}]


<br>

#### **2. TEXT-GENERATION**

In [15]:
from transformers import pipeline

# Hide download progress bars
logging.set_verbosity_error()

# Load the text generation pipeline
generator = pipeline("text-generation", model='gpt2')

# Provide a prompt to start the generation
prompt = "The most important skill for a data scientist is"
generated_text = generator(prompt, max_length=30)

print(generated_text)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[{'generated_text': 'The most important skill for a data scientist is to write a simple program that you can run on your computer. You can also use it to analyze data by making a few assumptions. For instance, you might think that you plan to write a program that says things like this:\n\nThis is the program with the most data.\n\nThis is the program with the most data. This is the program with the most data. This is the program with the most data. This is the program with the most data. This is the program with the most data.\n\nThe program with the most data might look something like this:\n\nThis is the program with the most data.\n\nThis is the program with the most data. This is the program with most data. This is the program with the most data. This is the program with the most data.\n\nThis is the program with the most data. This is the program with the most data. This is the program with the most data. This is the program with the most data. This is the program with the most da

<br>

#### **3. SUMMARIZATION**

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, logging
import torch

# Hide download progress bars
logging.set_verbosity_error()

# 1. Manual setup for summarization
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

long_text = """
The Hugging Face Hub is a platform with over 120,000 models, 20,000 datasets, and 50,000 demo apps (Spaces), all open source and publicly available, in an online platform where people can easily collaborate and build ML together.
"""

# 2. Process and generate
inputs = tokenizer(long_text, return_tensors="pt", max_length=1024, truncation=True)
summary_ids = model.generate(inputs["input_ids"], max_length=50, min_length=10, do_sample=False)
summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"Summary: {summary_text}")

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Summary:  The Hugging Face Hub is a platform with over 120,000 models, 20,000 datasets, and 50,000 demo apps (Spaces) The Hub is an online platform where people can easily collaborate and build ML together .


<br>

#### **4. QUESTION-ANSWERING**

In [13]:
!pip install -q torch

import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, logging

# Hide download progress bars
logging.set_verbosity_error()

# 1. Load model and tokenizer manually
model_name = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# 2. Define context and question
context = "My name is Adithya and I live in Hyderabad, India."
question = "Where do I live?"

# 3. Process inputs and get model output
inputs = tokenizer(question, context, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

# 4. Extract the answer from logits
answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
answer = tokenizer.decode(predict_answer_tokens)

print(f"Question: {question}")
print(f"Answer: {answer}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Question: Where do I live?
Answer: Hyderabad, India


#### **AUTO MODEL**

> AutoModel is a smart wrapper. You just give it the name of the model you want from the Hugging Face Hub, and it automatically reads the config.json file, figures out the correct underlying architecture, and loads the weights into memory.

In [20]:
import os
# 1. Disable progress bars globally before importing transformers
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from transformers import AutoTokenizer, AutoModel, logging
import torch

# 2. Suppress library warnings
logging.set_verbosity_error()

model_name = "distilbert-base-uncased"

# Dynamically load the correct tokenizer and base model architecture
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

text = "AutoClasses simplify model loading."

# Tokenize input text (convert to numbers)
inputs = tokenizer(text, return_tensors="pt")

# Pass through the base model
with torch.no_grad():
    outputs = model(**inputs)

# The base AutoModel outputs raw hidden states (embeddings)
embeddings = outputs.last_hidden_state

print(f"Shape of the embeddings [Batch Size, Sequence Length, Hidden Size]: {embeddings.shape}")
print("\nRaw mathematical representation of the first few tokens:")
print(embeddings[0, 0:3, 0:5])

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Shape of the embeddings [Batch Size, Sequence Length, Hidden Size]: torch.Size([1, 10, 768])

Raw mathematical representation of the first few tokens:
tensor([[-0.5503, -0.2936, -0.3398, -0.0335, -0.2995],
        [ 0.0460, -0.2066, -0.2378, -0.0365,  0.4183],
        [ 0.0999,  0.0758,  0.0541,  0.0948,  0.3516]])


<br>

#### **MODEL VS TOKENIZER**

> The Tokenizer (The Translator): Its job is to act as the bridge between human language and machine numbers. It chops your raw text into smaller pieces (words, subwords, or characters) called "tokens." It then looks up each token in its vocabulary and converts it into a specific integer ID. It also does the reverse: taking the model's numerical output and translating it back into human-readable text.

> The Model (The Brain): This is the actual neural network (like BERT or GPT). It takes the array of numerical IDs provided by the tokenizer, processes them through its many layers of weights and attention mechanisms, and outputs new numerical representations (either predictions, classifications, or the next tokens in a sequence).



```
[Raw Text Input: "I love GenAI"]
                      |
                      v
      +--------------------------------+
      |          TOKENIZER             | <-- Chops text into sub-words
      | (e.g., AutoTokenizer.encode)   | <-- Maps sub-words to integer IDs
      +--------------------------------+
                      |
                      v
      [Input IDs: [40, 2101, 3500, 99]]  <-- The mathematical representation
                      |
                      v
      +--------------------------------+
      |            MODEL               | <-- Applies attention mechanisms
      | (e.g., AutoModelForCausalLM)   | <-- Calculates probabilities
      +--------------------------------+
                      |
                      v
      [Output IDs: [40, 2101, 3500...]]  <-- Raw numerical predictions
                      |
                      v
      +--------------------------------+
      |          TOKENIZER             | <-- Looks up IDs in the vocabulary
      | (e.g., AutoTokenizer.decode)   | <-- Merges sub-words back together
      +--------------------------------+
                      |
                      v
     [Final Output: "I love GenAI too!"]
```



<br>

#### **INFERENCE PIPELINE**

> The pipeline() function in Hugging Face is so user-friendly because it completely hides the complexity of three distinct steps happening behind the scenes. When you run a single line of code like classifier("I love GenAI"), the pipeline automatically executes this entire workflow:

> The 3 Stages of the Inference Pipeline

1. Preprocessing: The raw input (like text, an image, or audio) is passed to a preprocessor (like a Tokenizer). This converts the human-readable data into the strict numerical format (tensors) that the model requires.

2. Model Forward Pass (Inference): The processed tensors are fed into the base model. The neural network calculates the math and outputs raw, unnormalized mathematical scores known as logits.

3. Postprocessing: Those raw logits are not very helpful to humans. The pipeline applies a final transformation (like a SoftMax function to calculate percentages) and maps the winning ID back to a human-readable label (like "POSITIVE" or "NEGATIVE").

<br>



```
[Raw Input: "This Hugging Face course is great!"]
                           |
                           v
           +--------------------------------+
           |  1. PREPROCESSING (Tokenizer)  | <-- Chops text, maps to IDs
           +--------------------------------+
                           |
                           v
      [Input Tensors: [[101, 2023, 7361, 102]]]
                           |
                           v
           +--------------------------------+
           |  2. MODEL FORWARD PASS         | <-- Passes tensors through the AI
           +--------------------------------+
                           |
                           v
           [Raw Logits: [[-1.56,  4.23]]]
                           |
                           v
           +--------------------------------+
           |  3. POSTPROCESSING             | <-- Applies Softmax (percentages)
           +--------------------------------+
                           |
                           v
  [Final Output: [{'label': 'POSITIVE', 'score': 0.99}]]
```





---



###### **ADITHYA UBALE**